### From protocol to pathway activity

In this tutorial, we will be diving into the protocols you have extracted. We will follow three main steps:
1. Integrate the concentration and duration of each growth factor into a compound-level score
2. Derive pathway scores based on compound action
3. Fit a multivariate logistic regression model to estimate the contribution of each pathway to the final cell type composition.

For the last step, we will be using the cell type quantification you derived from the scRNAseq data to find the connection between pathway activation and cell type differentiation.

To be efficient, we will do this over all protocols at once. Save all your extracted jsons in a folder and we will work with them all at once.

Let's first import our packages. As you have probably seen, the protocol information is saved in a file format called json. In this tutorial, you will learn how to parse it an extract the correct information.

In [ ]:
import pandas as pd
import json
import os

Based on the differentiation stage of interest, you have to decide which steps of the protocol to include in the analysis. For instance, if you wish to compare the pathway contributions that lead to foregut, hindgut and midgut, you will have to only include the steps (compounds and concentrations) that are used in differentiating cells until these stages. 

Let's begin by reading a json file containing one of your extractions into a variable called a dictionary. Later, we will process the entire folder.

In [2]:
path_to_protocol = 'data/protocols/27748754_Tiago.json'

with open(path_to_protocol) as json_data:
    prot = json.load(json_data)
    
prot

{'sequencingData': [{'accession': 'GSE81102', 'type': 'bulkRNAseq'}],
 'cellLines': [{'type': 'ESC', 'name': 'HES3'}, {'type': 'ESC', 'name': 'H9'}],
 'createdAt': '2026-03-11T21:06:10.986476+00:00',
 'steps': [{'duration': [],
   'coating': [],
   'passaging': {'enabled': False, 'media': []},
   'serumSupplements': [],
   'growthFactors': [],
   'stepNumber': 0,
   'readout': {'enabled': False, 'markers': []},
   'basalMedia': []},
  {'duration': [{'durationType': 'days', 'durationHours': '4'}],
   'coating': [],
   'passaging': {'enabled': False, 'media': []},
   'serumSupplements': [],
   'growthFactors': [{'name': 'BMP4',
     'unit': 'ng/mL',
     'concentration': '20',
     'vendor': 'R&D Systems',
     'catalogNumber': ''},
    {'name': 'VEGF',
     'unit': 'ng/mL',
     'concentration': '25',
     'vendor': 'PeproTech',
     'catalogNumber': ''},
    {'name': 'SCF',
     'unit': 'ng/mL',
     'concentration': '25',
     'vendor': 'PeproTech',
     'catalogNumber': ''},
    {'na

As you can see, there are a lot of different fields in the json. They contain the same information you have extracted in the online form:
- ```prot['cellLines']``` contains information about each starting cell line, such as ```type```, which is iPSC or ESC, and ```name```.
- ```prot['targetCell']``` contains the differentiation target you specified in the form
- ```prot['publicationId']``` contains the PMID of the publication

The actual protocol information is split per step in the ```steps``` field. For instance, to access the first step of the differentiation, we can do ```prot['steps'][1]```. Pay attention that the step numerotation follows the same logic as the online form, where step 0 is the stem cell culturing protocol, not the differentiation.

In [3]:
prot['steps'][1]

{'duration': [{'durationType': 'days', 'durationHours': '4'}],
 'coating': [],
 'passaging': {'enabled': False, 'media': []},
 'serumSupplements': [],
 'growthFactors': [{'name': 'BMP4',
   'unit': 'ng/mL',
   'concentration': '20',
   'vendor': 'R&D Systems',
   'catalogNumber': ''},
  {'name': 'VEGF',
   'unit': 'ng/mL',
   'concentration': '25',
   'vendor': 'PeproTech',
   'catalogNumber': ''},
  {'name': 'SCF',
   'unit': 'ng/mL',
   'concentration': '25',
   'vendor': 'PeproTech',
   'catalogNumber': ''},
  {'name': 'ACTIVIN A',
   'unit': 'ng/mL',
   'concentration': '10',
   'vendor': 'R&D Systems',
   'catalogNumber': ''},
  {'name': 'FGF2',
   'unit': 'ng/mL',
   'concentration': '10',
   'vendor': 'PeproTech',
   'catalogNumber': ''}],
 'stepNumber': 1,
 'readout': {'enabled': True,
  'markers': [{'geneEnrichment': 'up',
    'correspondingCellType': 'Mesoderm',
    'positiveCells': '',
    'name': 'MIXL1'},
   {'geneEnrichment': 'up',
    'correspondingCellType': 'Mesoderm',

Let's see how many steps we have in the protocol.

In [4]:
print ('There are', len(prot['steps']), 'including step 0')

There are 5 including step 0


You can also see that each step has different fields for basal media composition, serums and supplements, and growth factors. Let's print the list of growth factors in step 1.

In [5]:
for i in range(len(prot['steps'][1]['growthFactors'])):
    print (prot['steps'][1]['growthFactors'][i]['name'])

BMP4
VEGF
SCF
ACTIVIN A
FGF2


Feel free to explore more of the entries and compare them with the online form. 

Once you are familiar with the structure, we can move on to the next step, which is extracting the relevant information. In the next cell I am defining a few functions to help us do that:

- ```extract_growth_factors_into_df``` will parse the json until a chosen step and extract per step the duration, and the growth factor names and corresponding concentrations.
- ```compute_growth_factor_exposure``` groups the growth factors by name and publication and calculates the exposure of each compound, defined as the summed concentration x time per step.
- ```z_score_exposure```  Z-score scales the compound exposures among all publications to make them comparable.

In [16]:

def extract_growth_factors_into_df(protocol, untilStep, protocolID):
    """
    Extract step, duration (in days), growth factor name and concentration
    into a pandas DataFrame.

    Columns:
    - step
    - duration (in days, float)
    - growth_factor
    - concentration
    """
    rows = []

    for step in protocol.get("steps", []):
        step_number = step.get("stepNumber")

        if step_number is None or step_number == 0 or step_number > untilStep:
            continue

        # ---- Normalize duration to days ----
        duration_list = step.get("duration", [])
        duration_days = None

        if duration_list:
            durations = []
            for d in duration_list:
                value = d.get("durationHours")
                dtype = d.get("durationType")

                if value is None:
                    continue

                try:
                    value = float(value)
                except:
                    continue

                if dtype == "hours":
                    durations.append(value / 24)
                elif dtype == "days":
                    durations.append(value)
                else:
                    durations.append(value)  # fallback

            if durations:
                duration_days = sum(durations)  # total duration per step

        # ---- Extract growth factors ----
        growth_factors = step.get("growthFactors", [])

        if growth_factors:
            for gf in growth_factors:
                rows.append({
                    "step": step_number,
                    "duration": duration_days,
                    "growth_factor": gf.get("name"),
                    "concentration": gf.get("concentration")
                })
        else:
            rows.append({
                "step": step_number,
                "duration": duration_days,
                "growth_factor": None,
                "concentration": None
            })

    df = pd.DataFrame(rows)

    #Add PMID 
    df['protocol'] = protocolID

    return df

import pandas as pd
import numpy as np

def compute_growth_factor_exposure(df):
    """
    Compute exposure per (protocol, growth_factor) and z-score
    across protocols for each growth factor.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain:
        - 'protocol'
        - 'duration'
        - 'growth_factor'
        - 'concentration'

    Returns
    -------
    pandas.DataFrame
        Columns:
        - protocol
        - growth_factor
        - exposure
        - z_exposure (scaled per growth factor across protocols)
    """

   
    # Ensure numeric
    df["concentration"] = pd.to_numeric(df["concentration"], errors="coerce")

    # Drop invalid rows
    df = df.dropna(subset=["protocol", "growth_factor", "duration", "concentration"])

    # Compute exposure
    df["exposure"] = df["duration"] * df["concentration"]

    # Aggregate per protocol + growth factor
    df_grouped = (
        df.groupby(["protocol", "growth_factor"], as_index=False)["exposure"]
        .sum()
    )

    return df_grouped


def zscore(x):
    """ 
    Helper function to compute z score per input array
    """
   
    std = x.std()
    if std == 0 or np.isnan(std):
        return pd.Series([0] * len(x), index=x.index)
    return (x - x.mean()) / std


def z_score_exposure (df):
    
    # Z-score per growth factor across protocols
    def zscore(x):
        std = x.std()
        if std == 0 or np.isnan(std):
            return pd.Series([0] * len(x), index=x.index)
        return (x - x.mean()) / std

    df["z_exposure"] = (
        df.groupby("growth_factor")["exposure"]
        .transform(zscore)
    )

    return df

Let's first extract the growth factor information.

In [17]:
#if you have multiple protocols from the same publication, it is better to use something other than PMID as protocolID. 

growth_factor_df = extract_growth_factors_into_df(protocol=prot, untilStep=3, protocolID='27748754') #if you have multiple protocols, it is better to use a differnt ID to 
growth_factor_df.head()

,step,duration,growth_factor,concentration,protocol
0,1,4.0,BMP4,20,27748754
1,1,4.0,VEGF,25,27748754
2,1,4.0,SCF,25,27748754
3,1,4.0,ACTIVIN A,10,27748754
4,1,4.0,FGF2,10,27748754


Now let's calculate the scaled exposure values.

In [11]:
exposure_matrix = compute_growth_factor_exposure(growth_factor_df)
exposure_matrix.head()

,protocol,growth_factor,exposure
0,27748754,ACTIVIN A,80.0
1,27748754,BMP4,220.0
2,27748754,CHIR99021,12.0
3,27748754,FGF2,110.0
4,27748754,IGF2,90.0


## All protocols

To make the exposures comparable across different growth factors, we must scale them. We use Z-score scaling to convert the exposure values to mean 0 and unit variance for each compound acrosss all protocols. This process helps in reshaping the data as deviation from the mean. However, if we apply Z scaling to only one protocol, the value will essentially be zero. Therefore, it is now time to combine the data from all your protocols before we proceed.

However, before we can do that, we must decide the number of steps we take into consideration in our analysis, per protocol. Fill in the dictionary below with the correct step number for each protocol json file name.

In [18]:
dict_steps = {
    '27748754_Tiago.json' : 3,
    '36229560_Tiago.json' : 5, 
    '38676882_Tiago.json' : 2
    
}

In [ ]:
dict_steps = {
    'pathToJSON1.json' : 3,
    'pathToJSON2.json' : 5, 
    
}

As stated before, make sure you have all your protocols within one folder and adjust the path below accordingly. We will be looping through all the files in that folder, so make sure there are no other files other than the jsons in there. In essence, we read every protocol in the folder, process it and we concatenate the resulting exposure matrices at the end. 

In [22]:
path_to_protocols_folder = 'data/protocols' #adjust this with your actual path

#empty dictionary to store the resulting dataframes before concatenation
all_dfs = []

#Loop through all files in the specified folder
for path_to_protocol in os.listdir(path_to_protocols_folder):
    #Load the  json
    with open(os.path.join(path_to_protocols_folder, path_to_protocol)) as json_data:
        prot = json.load(json_data)
    
    #Get the final step from the dictionary defined above
    untilStep = dict_steps[path_to_protocol]

    #Extract growth factors and calculate exposure
    
    growth_factor_df = extract_growth_factors_into_df(prot, untilStep, path_to_protocol)
    exposure_df = compute_growth_factor_exposure(growth_factor_df)

     # Store
    all_dfs.append(exposure_df)

# Concatenate everything
final_df = pd.concat(all_dfs, ignore_index=True)

final_df.head()
    

C:\Users\rafae\AppData\Local\Temp\ipykernel_12080\746884217.py:109: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["exposure"] = df["duration"] * df["concentration"]
C:\Users\rafae\AppData\Local\Temp\ipykernel_12080\746884217.py:109: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["exposure"] = df["duration"] * df["concentration"]


,protocol,growth_factor,exposure
0,27748754_Tiago.json,ACTIVIN A,80.0
1,27748754_Tiago.json,BMP4,220.0
2,27748754_Tiago.json,CHIR99021,12.0
3,27748754_Tiago.json,FGF2,110.0
4,27748754_Tiago.json,IGF2,90.0


Now we can scale all the growth factors.

In [24]:
scaled_exposure_matrix = z_score_exposure (final_df)
scaled_exposure_matrix

,protocol,growth_factor,exposure,z_exposure
0,27748754_Tiago.json,ACTIVIN A,80.000000,0.000000
1,27748754_Tiago.json,BMP4,220.000000,0.707107
2,27748754_Tiago.json,CHIR99021,12.000000,0.000000
3,27748754_Tiago.json,FGF2,110.000000,0.000000
4,27748754_Tiago.json,IGF2,90.000000,0.000000
5,27748754_Tiago.json,SB431542,14.000000,0.000000
6,27748754_Tiago.json,SCF,350.000000,0.000000
7,27748754_Tiago.json,VEGF,350.000000,0.000000
8,36229560_Tiago.json,Ara-C,0.166667,0.000000
9,36229560_Tiago.json,SHH,816.666667,0.000000


## Pathway Clustering

The next step is now to cluster your growth factors into pathways. The way we do this is simple, we define a pathway as the sum of scaled growth factor exposure values that activate the pathway and subtract the sum of factors that inhibit the pathway. 

This requires you to define the pathways after consulting literature. Use the template below to code it. Let's start by taking a look at all the growth factors we have information about. Also take this opportunity to make sure the same compound is not present under different names in your pipeline (Ex. retinoic acid and RA). If so, correct it in the online form, download the updated json and run the code above again to process and concatenate all protocols.

In [25]:
print (scaled_exposure_matrix.growth_factor.unique())

['ACTIVIN A' 'BMP4' 'CHIR99021' 'FGF2' 'IGF2' 'SB431542' 'SCF' 'VEGF'
 'Ara-C' 'SHH' 'doxycycline']
